# ОТЧЕТ ПО МОДУЛЮ А: Проектирование архитектуры инфраструктуры данных

**Роль:** Ведущий аналитик транспортных данных  
**Конкурсант:** Анастасия  
**Дата:** 6.04.2026  
**Проект:** Интеллектуальная платформа мониторинга и прогнозирования транспортных потоков  

---

## 1. Анализ текущего состояния и требований (Критерий А1)

На основе брифа и бизнес-задач Департамента транспорта сформированы ключевые требования к проектируемой интеллектуальной платформе.

### 1.1 Функциональные требования
Система должна обеспечивать решение следующих бизнес-вопросов:

* **Мониторинг в реальном времени:** Расчет текущей загруженности дорог, средней скорости движения и фиксация структуры потока (доли легковых, грузовых ТС, автобусов, мотоциклов, велосипедов и пешеходов).
* **Предиктивная аналитика:** Построение прогноза загруженности участков и скоростного режима на горизонты 30, 60 и 120 минут для выработки прескриптивных рекомендаций диспетчерам.
* **Ретроспективный анализ и выявление паттернов:** Определение периодов (часы, дни недели, месяцы) с максимальными очередями и минимальными скоростями; выявление систематического появления большегрузов в часы пик.
* **Анализ безопасности:** Фиксация опасных сближений транспортных средств на основе координат рамок детекции и частоты таких событий.

### 1.2 Нефункциональные требования
* **Единая версия правды (Single Source of Truth):** Все подразделения должны работать с единым хранилищем (Datalake/DWH) без расхождений в цифрах, используя общие логические слои очистки и расчета метрик.
* **Автоматизация и Real-Time (SLA):** Полная автоматизация процессов от захвата видео до обновления дашбордов в реальном времени (задержка не более 2 минут). Исключение ручных операций.
* **Хранение истории (Историчность):** Обеспечение хранения полной сырой и агрегированной истории за последние 180 дней с использованием системных полей (например, `load_time`) для аудита и обучения ML-моделей.
* **Масштабируемость:** Гибкая архитектура, позволяющая подключать новые видеокамеры и внешние источники (погода, ДТП) без изменения существующего ядра системы.

---

## 2. Выбор архитектуры и технологий (Критерий А2)

Для реализации архитектуры был подобран следующий технологический стек. Каждая технология обоснована и строго привязана к конкретным требованиям системы.

| Компонент | Технология | Обоснование |
| :--- | :--- | :--- |
| **Детекция и трекинг** | YOLOv12s | Высокая точность и скорость обработки потокового видео в реальном времени. |
| **База данных (DWH)** | PostgreSQL | Надежное хранение структурированных данных и поддержка сложных аналитических запросов. |
| **ETL / Обработка** | Python (Pandas) | Гибкость трансформации данных и наличие библиотек для предиктивной аналитики. |
| **Оркестрация** | Apache Airflow | Автоматизация расписаний пакетной обработки и контроль зависимостей данных. |
| **BI / Визуализация** | Yandex DataLens | Бесшовная интеграция с DWH и поддержка интерактивных дашбордов. |

---

## 3. Проектирование архитектуры данных (Критерий А3)

### 3.1 Архитектурный подход
В качестве основного выбран **подход Ральфа Кимбалла (Kimball)** с ориентацией на бизнес-процессы (снизу-вверх) с использованием многослойной архитектуры.
* **Обоснование:** Подход предполагает создание нормализованного ядра и денормализованных витрин данных («звезда»), которые непосредственно отвечают на конкретные бизнес-вопросы. Это позволяет быстро поставлять ценность бизнесу. В перспективе заложена гибридная масштабируемость до уровня Data Vault.

### 3.2 Логические слои данных (Многослойная архитектура)
1.  **Raw Layer (Сырые данные):** Исходные детекции от нейросети YOLO и исторические выгрузки. Данные не изменяются и хранятся «как есть». Добавляется поле `load_time` для обеспечения историчности.
2.  **Integration / Core Layer (Базовый слой):** Очищенные данные. Удаление дубликатов, стандартизация форматов, связывание Track ID и расчет базовых метрик.
3.  **Data Mart Layer (Витринный слой):** Подготовленные витрины для Yandex DataLens (агрегаты в БД или CSV).

### 3.3 Потоки данных
В системе реализована **Lambda-подобная архитектура**:
* **Real-time поток:** Видеопоток → YOLO → Слой 1 → Слой 2 (Мгновенный расчет) → Слой 3 (Real-time витрина) → DataLens.
* **Пакетный поток (Batch):** История/Накопленные данные → Airflow (ETL) → Очистка → ML-прогнозирование → Слой 3 (Витрины истории и прогнозов).

---

## 4. Модели транспортных данных для аналитики (Критерий А4)

### 4.1 Концептуальная модель
Выделены следующие корневые сущности:
* **Камера:** ID, координаты, статус.
* **Видеопоток:** привязка к камере, FPS, интервал.
* **Детекция (Событие):** метка времени, координаты (BBox), уверенность.
* **Транспортное средство:** классификация (класс объекта).
* **Track ID:** сквозной ID для трекинга ТС между кадрами.

### 4.2 Физическая модель данных по слоям

#### Слой Raw / Core (Таблица `detections`):
* `frame` (int) — номер кадра.
* `track_id` (int) — уникальный ID ТС.
* `time_ms` (int) — временная метка.
* `class` (varchar) — тип ТС.
* `confidence` (float) — точность детекции.
* `x1, y1, x2, y2` (int) — координаты Bounding Box.
* `load_time` (timestamp) — время загрузки в БД.

#### Data Mart Layer (Витрины данных):

1.  **Витрина 1: Интенсивность**
    * *Вопрос:* Сколько машин проезжает в минуту/час?
    * *Поля:* `minute` (PK), `vehicles_count`.
    * *Обновление:* Real-time.
2.  **Витрина 2: Состав потока**
    * *Вопрос:* Доля различных типов ТС?
    * *Поля:* `class` (PK), `unique_vehicles`, `percentage`.
    * *Обновление:* Daily Batch.
3.  **Витрина 3: Средняя скорость**
    * *Вопрос:* Скорость по категориям ТС?
    * *Поля:* `class` (PK), `avg_speed`.
    * *Обновление:* Real-time.
4.  **Витрина 4: Прогноз загруженности**
    * *Вопрос:* Загруженность через 30/60/120 минут?
    * *Поля:* `timestamp` (PK), `horizon_30min`, `horizon_60min`, `horizon_120min`.
    * *Обновление:* Micro-batch (каждые 5 мин).

---

## 5. Заключение
Спроектированная архитектура полностью отвечает требованиям заказчика. **Единая версия правды** обеспечивается централизацией в Raw-слое. Использование системных меток времени позволяет хранить историю за 180 дней для ML-задач. Изоляция витрин по бизнес-сущностям гарантирует высокую производительность визуализации в Yandex DataLens.